# Морфоанализ, омонимы, склонение и сборка SEO-названий через API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DimaKorshikov/rulemma/blob/main/examples/notebooks/03_morphology_inflection_and_seo_titles.ipynb)

Практический разбор грамматических методов HTTP API `rulemma`: `POST /v1/analyze` (часть речи, род, число, падежи, все кандидаты омонимов, OOV-вывод `inferred` и управление предлогов `prep_governs`), `POST /v1/inflect` (склонение коротких товарных фраз с защитой брендов, фонетические предлоги и согласование с числом) и `POST /v1/seoname`.

### Регистрация и доступ к HTTP API

- Личный кабинет и бесплатный API-ключ: **[rulemma.ru/account/](https://rulemma.ru/account/)** (вход по email, Яндекс ID или Google)
- Документация HTTP API: [rulemma.ru/api/](https://rulemma.ru/api/) · Тарифы и лимиты: [rulemma.ru/pricing/](https://rulemma.ru/pricing/) · OpenAPI: [openapi.yaml](https://rulemma.ru/openapi.yaml)
- Базовый адрес API: `https://rulemma.ru/v1` · Авторизация: заголовок `Authorization: Bearer rlm_live_...` (передавать ключ в URL запрещено)

Как подключить ключ:
1. Создайте бесплатный аккаунт на [rulemma.ru/account/](https://rulemma.ru/account/) и скопируйте токен `rlm_live_...`.
2. Сохраните его в переменную окружения `RULEMMA_API_KEY` (в Google Colab удобно положить токен в панель `Secrets` под именем `RULEMMA_API_KEY` или ввести в ячейке настройки).
3. Без ключа все 3 метода в тетрадке автоматически выполняются через песочницу `/v1/sandbox/test` (до 10 строк); с токеном с [rulemma.ru/account/](https://rulemma.ru/account/) работают полные батчи.

### Содержание тетрадки

1. Подключение к API `rulemma.ru`
2. Морфологический разбор, омонимы и OOV-вывод (`POST /v1/analyze`)
3. Проверка предложного управления `prep_governs` («с цинком» vs «витамин С 1000»)
4. Склонение товарных фраз, предлоги и согласование с числом (`POST /v1/inflect`)
5. Сборка и согласование SEO-названий карточек (`POST /v1/seoname`)

## 1. Клиент HTTP API

In [ ]:
import json
import os
import urllib.error
import urllib.request

BASE_URL = os.environ.get("RULEMMA_API_BASE", "https://rulemma.ru/v1").rstrip("/")
API_KEY = os.environ.get("RULEMMA_API_KEY", "").strip()

# В Google Colab ключ можно автоматически прочитать из вкладки Secrets
if not API_KEY:
    try:
        from google.colab import userdata  # type: ignore

        API_KEY = (userdata.get("RULEMMA_API_KEY") or "").strip()
    except Exception:
        pass

HAS_KEY = bool(API_KEY)
print("Базовый URL:", BASE_URL)
print(
    "Режим:",
    "боевой API с Bearer-токеном" if HAS_KEY else "публичная песочница без ключа (до 10 строк)",
)
if not HAS_KEY:
    print("Получить бесплатный API-ключ: https://rulemma.ru/account/")


def api_request(
    method: str,
    path: str,
    payload: dict | None = None,
    *,
    auth: bool = True,
    timeout: int = 20,
) -> tuple[dict, dict[str, str]]:
    """Выполняет HTTP-запрос к rulemma API без сторонних зависимостей."""
    url = f"{BASE_URL}{path}"
    headers = {
        "Accept": "application/json",
        "User-Agent": "rulemma-jupyter-example/1.0",
    }
    if auth:
        if not API_KEY:
            raise RuntimeError(
                "Требуется RULEMMA_API_KEY. Выпустите бесплатный токен на https://rulemma.ru/account/"
            )
        headers["Authorization"] = f"Bearer {API_KEY}"

    body = None
    if payload is not None:
        body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json; charset=utf-8"

    import time as _time

    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    for attempt in range(2):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                resp_headers = {k.lower(): v for k, v in resp.headers.items()}
                return json.loads(resp.read().decode("utf-8")), resp_headers
        except urllib.error.HTTPError as exc:
            raw = exc.read().decode("utf-8", errors="replace")
            if exc.code == 429 and attempt == 0:
                wait = min(12, max(1, int(exc.headers.get("Retry-After", "4"))))
                print(f"Лимит скорости (HTTP 429), пауза {wait} с...")
                _time.sleep(wait)
                continue
            raise RuntimeError(f"HTTP {exc.code} {path}: {raw}") from exc
    raise RuntimeError(f"Не удалось выполнить {method} {path}")

## 2. Морфоанализ слов, омонимы и OOV-вывод (`POST /v1/analyze`)

Поле `candidates` возвращает все словарные разборы омонима (например, `пил` → глагол `пить`, сущ. `пил`, сущ. `пила`), а `inferred: true` помечает формы, выведенные по парадигмам вне базовой таблицы.

In [ ]:
words = ["пил", "стали", "мелки", "дипломом", "кофемашина"]

if HAS_KEY:
    resp, _ = api_request("POST", "/analyze", {"words": words})
else:
    resp, _ = api_request(
        "POST",
        "/sandbox/test",
        {"method": "analyze", "words": words},
        auth=False,
    )

for row in resp["items"]:
    cands = ", ".join(f"{c['lemma']}:{c['pos']}" for c in (row.get("candidates") or []))
    tag = "inferred" if row.get("inferred") else row.get("source", "")
    print(
        f"{row['word']:12s} -> {row['lemma']:12s} | pos={row.get('pos', '-'):4s} "
        f"| gender={(row.get('gender') or '-'):4s} | cases={row.get('cases', [])} "
        f"| [{tag}]"
    )
    if cands:
        print(f"               кандидаты: {cands}")

## 3. Предложное управление `prep_governs`

Отличает настоящий предлог (`витамины с цинком`, `специалист с дипломом`) от буквы-маркера серии или дозировки (`витамин с 1000`, `класс в`).

In [ ]:
next_words = ["цинком", "дипломом", "1000", "д"]

if HAS_KEY:
    gov, _ = api_request("POST", "/analyze", {"words": next_words, "prep": "с"})
else:
    gov, _ = api_request(
        "POST",
        "/sandbox/test",
        {"method": "analyze", "words": next_words, "prep": "с"},
        auth=False,
    )

for row in gov["items"]:
    print(f"предлог 'с' + {row['word']!r:12s} -> prep_governs={row.get('prep_governs')}")

## 4. Склонение именных групп, предлоги и согласование с числом (`POST /v1/inflect`)

Бренды, латинские артикулы, предложные хвосты и несклоняемые слова остаются нетронутыми, а ядро фразы корректно склоняется в нужный падеж (`nomn`, `gent`, `datv`, `accs`, `ablt`, `loct`).

In [ ]:
phrases = [
    "стиральная машина LG F2V5",
    "чехол для телефона",
    "Краснодарский край",
    "зимние ножницы",
]

cases_to_run = [("accs", "", None), ("loct", "в", None), ("gent", "", 21)] if HAS_KEY else [("accs", "", None), ("loct", "в", 21)]
for target_case, prep, count in cases_to_run:
    req_body = {"phrases": phrases, "case": target_case}
    if prep:
        req_body["prep"] = prep
    if count is not None:
        req_body["count"] = count

    if HAS_KEY:
        inf, _ = api_request("POST", "/inflect", req_body)
    else:
        inf, _ = api_request(
            "POST",
            "/sandbox/test",
            {"method": "inflect", **req_body},
            auth=False,
        )

    print(f"=== Падеж {target_case} (prep={prep!r}, count={count}) ===")
    for row in inf["items"]:
        extra = ""
        if row.get("with_prep"):
            extra += f" | с предлогом: {row['with_prep']}"
        if row.get("agree_number"):
            extra += f" | с числом: {row['agree_number']}"
        print(f"  {row['input']:28s} -> {row['result']}{extra}")

## 5. Сборка и согласование SEO-названий карточек (`POST /v1/seoname`)

Добавляет недостающие свойства к базовому названию товара, согласует прилагательные и причастия с родом и числом типа товара, отсекает дубли и соблюдает лимит `max_len`.

In [ ]:
cards = [
    {
        "base_name": "Сумка кожаная",
        "product_type": "Сумка",
        "attributes": ["кожаный", "синие", "женские", "вместительный"],
        "max_len": 100,
    },
    {
        "base_name": "Ведро",
        "product_type": "Ведро",
        "attributes": ["оцинкованные", "строительный", "12 л"],
        "max_len": 80,
    },
    {
        "base_name": "Купальники",
        "product_type": "Купальники",
        "attributes": ["женский", "раздельный", "чёрный"],
        "max_len": 80,
    },
]

if HAS_KEY:
    seo_resp, hdrs = api_request("POST", "/seoname", {"items": cards})
    results = seo_resp["items"]
    print(f"Списано единиц: {seo_resp['units_charged']} (остаток: {hdrs.get('x-quota-remaining', '?')})")
else:
    results = []
    demo_cards = cards[:2]
    for card in demo_cards:
        sbx, _ = api_request("POST", "/sandbox/test", {"method": "seoname", **card}, auth=False)
        results.append(sbx["items"])
    cards = demo_cards
    print("Режим песочницы (2 карточки по одной; с ключом доступен батч items[]):")

for card, res in zip(cards, results):
    print(f"  {card['base_name']:16s} -> {res['seo_name']}  (добавлено={res['added']}, согласовано={res['agreed']})")

---

## Что дальше

- Следующая тетрадка: [04_text_frequency_and_tfidf_api.ipynb](04_text_frequency_and_tfidf_api.ipynb)
- Все Jupyter-тетрадки с HTTP API: [examples/notebooks/README.md](README.md)
- Регистрация и управление API-ключами: [rulemma.ru/account/](https://rulemma.ru/account/)
- Документация эндпоинтов и форматов ошибок: [rulemma.ru/api/](https://rulemma.ru/api/) · [openapi.yaml](https://rulemma.ru/openapi.yaml)
- Нужны миллионы строк локально без сети (Go / Python CGO / CLI)? Смотрите локальные примеры [examples/01..13](../README.md).